# **Banff Working Group Code**

Goal: Use Celia's existing code (to take a (finite) rectangle module and extract its critical values) and make matching distance more precise by computing lines instead of sampling lines along a grid. 

## Import all the things.


In [1]:
import gudhi
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import numpy as np
import matplotlib
from math import cos, sin, pi, atan
import csv # this is for reading input files

import sys 
sys.path.append('../Code/')
import modules as mod

# This reloads external modules every time a cell is run
# so we don't have to reboot kernel every time we modify classes.py
%load_ext autoreload
%autoreload 2

# Functions

In [22]:
# importer function to run over rectangle rect mod (.rmod) data file

def import_rmod(filename):
    # here we import a file with a list of rectangles and the modules they belong to. 
    data = []
    datalist = {}
    templist = {}
    rectangles = {}
    # ERROR CHECKING
    try: fd = open(filename, 'r')
    except OSError:
        print("Could not open/read file")
        sys.exit()
    with fd:
        reader = csv.reader(fd, delimiter=':')
        num_entries = 0
        for row in reader:
            if not row:
                print("error! row not found!")
            num_entries = num_entries + 1
            data.append(list(row))
            datalist = []
            rectangles = []
        # LINE BY LINE READ IN 
        i = 0
        for x in data:
            temprectx1y1 = [float(item) for item in data[i][0].split()]
            temprectx2y2 = [float(item) for item in data[i][1].split()]
            tempdim = [int(item) for item in data[i][2].split()]
            tempid = [int(item) for item in data[i][3].split()]
            templist = [temprectx1y1, temprectx2y2, tempdim, tempid]
            temprectangle = mod.Rectangle(temprectx1y1,temprectx2y2,tempdim)
            datalist.append(templist)
            rectangles.append(temprectangle)
            i += 1
            
        # all_rectangles is a list with x1y1, x2y2, dim, parent module
        # TODO: we shuffle the same data between variabes; this could be cleaned
        all_rectangles = datalist
        # for every line in all_rectangles, make a mod.Rectangle, 
        # and then rmod.addrect() to the module in the ith position in 
        # a module list, corresponding to the parent id.
        rects_list = []
        r_mods_list = [None] * len(all_rectangles)
        # this is the max possible size it could be, but prob smaller
        # TODO: double check that this does the right thing
        i = 0
        for x in all_rectangles:
            temp_rect = mod.Rectangle(x[0], x[1], x[2][0], x[3][0])
            rects_list.append(temp_rect)
            i += 1
        j = 0
        for y in rects_list:
            if r_mods_list[y.find_parent()] == None:
                # we need to initialize
                r_mods_list[y.find_parent()] = mod.RectangleModule([y])
            else:
                # this mod already initialized. append. 
                r_mods_list[y.find_parent()].add_rectangle(y)
                # should be [y]?!
            j += 1
        # this is a sneaky way using list comprehension to remove Nones
        r_mods_list = [i for i in r_mods_list if i]
    return r_mods_list

In [34]:
r_mods_list = import_rmod('../data/sample.rmod')
i = 0
for z in r_mods_list:
    print("Rectangle module", i , "info: ")
    z.print_mod_info()
    
    i += 1
# stopping point: right now, r_mods_list
# is a list of all of the modules from the rmod file. 
# next: incorporate this into function, and add printing
# to the rmods class in the modules file for error checking




Rectangle module 0 info: 
total rectangles: 2 . 

bottom left: [0.0, -1.0]
top right: [7.0, 6.0]
dimension: 1
parent mod: 0 

bottom left: [0.0, 3.0]
top right: [7.0, 10.0]
dimension: 1
parent mod: 0 

Rectangle module 1 info: 
total rectangles: 2 . 

bottom left: [0.0, -1.0]
top right: [7.0, 10.0]
dimension: 1
parent mod: 1 

bottom left: [0.0, 3.0]
top right: [7.0, 6.0]
dimension: 1
parent mod: 1 



# TO DO 

### ERROR CHECKING, importer func + mod class: 
- mod.rectmod : check that all dimensions are the same in a mod
- remove duplicate rectangles if any, or at least warn
- check that numbers make sense, like dim is + 
- also check sensible numbers in mod.rect 
- Question: could two parent modules share same rectangle?